# 01 — Exploratory data analysis: IBM AML HI-Small

Each section follows **question → code → output → interpretation**. Decisions that come out of a section move into `src/` as functions with tests; this notebook keeps the evidence and the reasoning.

**Public repo rule:** outputs show aggregates and plots only, never raw rows.

## 0. Setup

In [1]:
import pandas as pd

from skeptical_second_reader.data import ID_COLUMNS, load_transactions

## 1. Load

**Question:** does the loader keep the data intact? Two traps in the raw file: both account columns are named `Account`, and bank codes carry leading zeros (`010`), which the pandas pyarrow engine silently drops.

The first call reads the CSV (slow) and caches it as parquet in `data/interim/`; later calls read the parquet.

In [2]:
%%time
df = load_transactions()
print(f"rows: {len(df):,}  columns: {df.shape[1]}")

rows: 5,078,345  columns: 11
CPU times: total: 15.2 s
Wall time: 15.6 s


In [3]:
df.dtypes

timestamp             datetime64[us]
from_bank                        str
from_account                     str
to_bank                          str
to_account                       str
amount_received              float64
receiving_currency               str
amount_paid                  float64
payment_currency                 str
payment_format                   str
is_laundering                  int64
dtype: object

In [4]:
# Invariants of the loaded data; the notebook stops here if one fails.
assert df.isna().sum().sum() == 0, "missing values"
assert set(df["is_laundering"].unique()) <= {0, 1}, "label is not binary"
for col in ID_COLUMNS:
    assert pd.api.types.is_string_dtype(df[col]), f"{col} is not a string column"
assert df["from_bank"].str.startswith("0").any(), "leading zeros were dropped"

print("time span:", df["timestamp"].min(), "->", df["timestamp"].max())
print(f"laundering rate: {df['is_laundering'].mean():.4%}")

time span: 2022-09-01 00:00:00 -> 2022-09-18 16:18:00
laundering rate: 0.1019%


**Interpretation:** The data loaded correctly: all invariant checks passed (no missing values, binary label, ID columns stored as strings with leading zeros intact). The dataset contains 5,078,345 transactions covering about 18 days (1–18 September 2022). The laundering rate is 0.1019%, roughly 1 in 980 transactions, so the classes are highly imbalanced. A model that labels every transaction as clean would reach about 99.9% accuracy while catching no laundering at all. Accuracy is therefore not informative here; we will use AUC-PR (area under the precision–recall curve) as the main metric.